In [0]:
dbutils.widgets.text("epic_secrets_scope", "", "Epic Secrets Scope")
dbutils.widgets.text("client_id_dbs_key", "", "Epic Client ID DB Secrets Key")
dbutils.widgets.text("token_url", "", "Epic Token URL")
dbutils.widgets.text("algo", "", "Epic Token Enrcyption Algorithm")

In [0]:
widget_values = dbutils.widgets.getAll()
for name, value in widget_values.items():
	print(f"{name} = {value}")

In [0]:
import time
import jwt
import requests

In [0]:
CLIENT_ID = dbutils.secrets.get(scope = widget_values["epic_secrets_scope"], key = widget_values["client_id_dbs_key"])
TOKEN_URL = widget_values["token_url"]
PRIVATE_KEY = dbutils.secrets.get(scope = widget_values["epic_secrets_scope"], key = "private_key")
ALGO = widget_values["algo"]
EPIC_KID = dbutils.secrets.get(scope=widget_values["epic_secrets_scope"], key="kid")

In [0]:
print(f"""
CLIENT_ID = {CLIENT_ID}
TOKEN_URL = {TOKEN_URL}
PRIVATE_KEY = {PRIVATE_KEY}
ALGO = {ALGO}
EPIC_KID = {EPIC_KID}
""")

In [0]:
def get_access_token():

	now = int(time.time())
	payload = {
		"iss": CLIENT_ID
		, "sub": CLIENT_ID
		, "aud": TOKEN_URL
		, "jti": str(now)
		, "exp": now + 300  # Token valid for 5 minutes
	}
	
	jwt_headers = {
        "kid": EPIC_KID,
        "typ": "JWT"
	}

	jwt_token = jwt.encode(payload, PRIVATE_KEY, algorithm=ALGO, headers=jwt_headers)

	headers = { "Content-Type": "application/x-www-form-urlencoded" }
	data = {
		"grant_type": "client_credentials"
		, "client_assertion_type": "urn:ietf:params:oauth:client-assertion-type:jwt-bearer"
		, "client_assertion": jwt_token
		# , "scope": "system/Patient.read"
	}

	response = requests.post(TOKEN_URL, data=data, headers=headers)
	response.raise_for_status()
	# return response.json()["access_token"]
	return response.json()

In [0]:
import json

token = get_access_token()
print(json.dumps(token, indent=2))

In [0]:
token = get_access_token()
headers = {
	"Authorization": f"Bearer {token['access_token']}"
    ,"Accept": "application/json"
}
url = "https://fhir.epic.com/interconnect-fhir-oauth/api/FHIR/R4/Patient/T1wI5bk8n1YVgvWk9D05BmRV0Pi3ECImNSK8DKyKltsMB"
response = requests.get(url, headers=headers)
response.raise_for_status()
print(response.status_code, response.text)

In [0]:
response.json()

In [0]:
"""
Class to handle authentication
"""

import datetime, jwt, requests, json, zoneinfo
from uuid import uuid4

class EpicApiAuth(requests.auth.AuthBase):
  def __init__(self, 
               client_id, 
               private_key, 
               kid,
               algo,
               auth_location = "https://fhir.epic.com/interconnect-fhir-oauth/oauth2/token"):
    self.__client_id = client_id
    self.__private_key = private_key
    self.__kid = kid
    self.__algo = algo
    self.auth_location = auth_location
    self.__token = None
    self.token_expiry = None

  def get_token(self,
                now = None,
                expiration = None,
                timeout=30):
    now = (now if now is not None else datetime.datetime.now(zoneinfo.ZoneInfo("America/New_York")))
    expiration = (expiration if expiration is not None else datetime.datetime.now(zoneinfo.ZoneInfo("America/New_York")) + datetime.timedelta(minutes=5))
    if self.__token is None or now >= self.token_expiry:
      t = self.generate_token(now, expiration, timeout)
      t.raise_for_status()
      self.__token = json.loads(t.text)
      self.token_expiry = expiration
    return self.__token
  
  def __call__(self, r):
    r.headers['Authorization'] = 'Bearer %s' % self.get_token()['access_token']
    r.headers['Accept'] = 'application/json'
    return r

  """
    Provide authentication to EPIC on FHIR OAuth2 and return valid token
      @param expiration = the datetime when the token expires, default 5 minutes
      @param timeout = seconds to timeout request, default 30 
  """
  def generate_token(self,
                     now = None,
                     expiration = None,
                     timeout=30):
    now = (now if now is not None else datetime.datetime.now(zoneinfo.ZoneInfo("America/New_York")))
    expiration = (expiration if expiration is not None else datetime.datetime.now(zoneinfo.ZoneInfo("America/New_York")) + datetime.timedelta(minutes=5))
    return requests.post(self.auth_location, 
        data= {
        'grant_type': 'client_credentials',
        'client_assertion_type': 'urn:ietf:params:oauth:client-assertion-type:jwt-bearer',
        'client_assertion': jwt.encode(
           {
              'iss': self.__client_id,
              'sub': self.__client_id,
              'aud': self.auth_location,
              'exp': int(expiration.timestamp()),
              'iat': int(now.timestamp()),
              'jti': uuid4().hex,
          },
          self.__private_key,
          algorithm=self.__algo,
          headers={
            'kid': self.__kid,
            'alg': self.__algo,
            'typ': 'JWT',
          })
      }, timeout=timeout)
    
  def can_connect(self):
    return (self.generate_token().status_code == 200)

In [0]:
epicAuth = EpicApiAuth(
  client_id = CLIENT_ID,
  private_key = PRIVATE_KEY,
  kid = EPIC_KID,
  algo = ALGO,
  auth_location = TOKEN_URL)

In [0]:
epicAuth.can_connect()

In [0]:
class EpicApiRequest:
    def __init__(self, auth, base_url):
        self.auth = auth
        self.base_url = base_url

    def make_request(self, http_method, resource, action, data=None):
        response = getattr(requests, http_method)(f"{self.base_url}{resource}/{action}", data=data, auth=self.auth)
        
        return {'request':
                {
                    'http_method': http_method,
                    'url': f"{self.base_url}{resource}/{action}",
                    'data': ('' if data is None else data)
                },
                'response': {
                    'response_status_code': response.status_code, 
                    'response_time_seconds': (response.elapsed.microseconds / 1000000),
                    'response_headers': response.headers,
                    'response_text': response.text,
                    'response_url': response.url
                }
            }

In [0]:
epic_api = EpicApiRequest(
    auth = epicAuth,
    base_url = "https://fhir.epic.com/interconnect-fhir-oauth/api/FHIR/R4/"
)

In [0]:
epic_api.make_request(
    http_method = "get",
    resource = "Patient",
    action = "T1wI5bk8n1YVgvWk9D05BmRV0Pi3ECImNSK8DKyKltsMB"
)